# ETL Gold - Carga de Datos al Modelo Dimensional

**Prerrequisito**: Ejecutar `DDL_Gold` la primera vez para crear esquema y tablas.

**Propósito**: Cargar datos desde Silver a las tablas Gold (4 dimensiones + 1 fact).

**Flujo**:
1. Carga dim_marca
2. Carga dim_tipo_vehiculo
3. Carga dim_modelo
4. Carga dim_geografia
5. Carga fact_transferencias
6. Validaciones de calidad

**Nota**: Usa INSERT OVERWRITE para reemplazar datos completamente en cada ejecución.

In [0]:
%sql
-- Cargar dimensión: dim_marca
-- La estructura de la tabla fue creada por DDL_Gold
-- PK: automotor_marca_codigo
INSERT OVERWRITE TABLE workspace.tp_dnrpa_gold.dim_marca
SELECT DISTINCT
  automotor_marca_codigo,
  automotor_marca_descripcion
FROM workspace.tp_dnrpa_silver.silver_transferencias
WHERE automotor_marca_codigo IS NOT NULL
ORDER BY automotor_marca_codigo;

In [0]:
%sql
-- Cargar dimensión: dim_tipo_vehiculo
-- La estructura de la tabla fue creada por DDL_Gold
-- PK: automotor_tipo_codigo
INSERT OVERWRITE TABLE workspace.tp_dnrpa_gold.dim_tipo_vehiculo
SELECT DISTINCT
  automotor_tipo_codigo,
  automotor_tipo_descripcion
FROM workspace.tp_dnrpa_silver.silver_transferencias
WHERE automotor_tipo_codigo IS NOT NULL
ORDER BY automotor_tipo_codigo;

In [0]:
%sql
-- Cargar dimensión: dim_modelo
-- La estructura de la tabla fue creada por DDL_Gold
-- PK: automotor_modelo_codigo
INSERT OVERWRITE TABLE workspace.tp_dnrpa_gold.dim_modelo
SELECT DISTINCT
  automotor_modelo_codigo,
  automotor_modelo_descripcion
FROM workspace.tp_dnrpa_silver.silver_transferencias
WHERE automotor_modelo_codigo IS NOT NULL
ORDER BY automotor_modelo_codigo;

In [0]:
%sql
-- Cargar dimensión: dim_geografia
-- La estructura de la tabla fue creada por DDL_Gold
-- PK: registro_seccional_codigo
-- Fix: Filtra registros con provincia NULL para evitar duplicados
INSERT OVERWRITE TABLE workspace.tp_dnrpa_gold.dim_geografia
SELECT DISTINCT
  registro_seccional_codigo,
  registro_seccional_provincia
FROM workspace.tp_dnrpa_silver.silver_transferencias
WHERE registro_seccional_codigo IS NOT NULL
  AND registro_seccional_provincia IS NOT NULL
  AND registro_seccional_provincia != 'NULL'
ORDER BY registro_seccional_codigo;

In [0]:
%sql
-- Cargar tabla de hechos: fact_transferencias
-- La estructura de la tabla fue creada por DDL_Gold
-- PK: id_tramite
-- FKs: automotor_marca_codigo, automotor_tipo_codigo, automotor_modelo_codigo, registro_seccional_codigo
INSERT OVERWRITE TABLE workspace.tp_dnrpa_gold.fact_transferencias
SELECT
  -- Primary Key
  id_tramite,
  
  -- Fecha del trámite
  tramite_fecha,
  
  -- Métricas cuantitativas
  automotor_anio_modelo,
  
  -- Foreign Keys a las dimensiones (solo códigos)
  automotor_marca_codigo,
  automotor_tipo_codigo,
  automotor_modelo_codigo,
  registro_seccional_codigo
  
FROM workspace.tp_dnrpa_silver.silver_transferencias
WHERE id_tramite IS NOT NULL;

## Validaciones de Calidad del Modelo Dimensional

Esta sección verifica la integridad y calidad de los datos en la capa Gold:

* **Conteos de registros**: Verificar que las tablas se crearon correctamente
* **Unicidad de PKs**: Asegurar que las claves primarias sean únicas
* **Integridad referencial**: Verificar que todas las FKs existan en sus dimensiones
* **Valores nulos**: Detectar problemas de calidad en columnas críticas
* **Reconciliación**: Comparar conteos entre Silver y Gold

In [0]:
%sql
-- Validación 1: Conteo de registros en todas las tablas del modelo
-- Permite verificar que las tablas se poblaron correctamente

SELECT 'dim_marca' AS tabla, COUNT(*) AS total_registros FROM workspace.tp_dnrpa_gold.dim_marca
UNION ALL
SELECT 'dim_tipo_vehiculo' AS tabla, COUNT(*) AS total_registros FROM workspace.tp_dnrpa_gold.dim_tipo_vehiculo
UNION ALL
SELECT 'dim_modelo' AS tabla, COUNT(*) AS total_registros FROM workspace.tp_dnrpa_gold.dim_modelo
UNION ALL
SELECT 'dim_geografia' AS tabla, COUNT(*) AS total_registros FROM workspace.tp_dnrpa_gold.dim_geografia
UNION ALL
SELECT 'fact_transferencias' AS tabla, COUNT(*) AS total_registros FROM workspace.tp_dnrpa_gold.fact_transferencias
ORDER BY tabla;

In [0]:
%sql
-- Validación 2: Verificar unicidad de Primary Keys en dimensiones
-- Si hay duplicados, el conteo de PKs distintas será menor al total de registros

SELECT 
  'dim_marca' AS tabla,
  COUNT(*) AS total_registros,
  COUNT(DISTINCT automotor_marca_codigo) AS pk_distintas,
  CASE WHEN COUNT(*) = COUNT(DISTINCT automotor_marca_codigo) THEN '✓ OK' ELSE '✗ DUPLICADOS' END AS resultado
FROM workspace.tp_dnrpa_gold.dim_marca

UNION ALL

SELECT 
  'dim_tipo_vehiculo' AS tabla,
  COUNT(*) AS total_registros,
  COUNT(DISTINCT automotor_tipo_codigo) AS pk_distintas,
  CASE WHEN COUNT(*) = COUNT(DISTINCT automotor_tipo_codigo) THEN '✓ OK' ELSE '✗ DUPLICADOS' END AS resultado
FROM workspace.tp_dnrpa_gold.dim_tipo_vehiculo

UNION ALL

SELECT 
  'dim_modelo' AS tabla,
  COUNT(*) AS total_registros,
  COUNT(DISTINCT automotor_modelo_codigo) AS pk_distintas,
  CASE WHEN COUNT(*) = COUNT(DISTINCT automotor_modelo_codigo) THEN '✓ OK' ELSE '✗ DUPLICADOS' END AS resultado
FROM workspace.tp_dnrpa_gold.dim_modelo

UNION ALL

SELECT 
  'dim_geografia' AS tabla,
  COUNT(*) AS total_registros,
  COUNT(DISTINCT registro_seccional_codigo) AS pk_distintas,
  CASE WHEN COUNT(*) = COUNT(DISTINCT registro_seccional_codigo) THEN '✓ OK' ELSE '✗ DUPLICADOS' END AS resultado
FROM workspace.tp_dnrpa_gold.dim_geografia

UNION ALL

SELECT 
  'fact_transferencias' AS tabla,
  COUNT(*) AS total_registros,
  COUNT(DISTINCT id_tramite) AS pk_distintas,
  CASE WHEN COUNT(*) = COUNT(DISTINCT id_tramite) THEN '✓ OK' ELSE '✗ DUPLICADOS' END AS resultado
FROM workspace.tp_dnrpa_gold.fact_transferencias;

In [0]:
%sql
-- Validación 3: Detectar valores nulos en Primary Keys y Foreign Keys
-- Estas columnas NO deben contener nulos

SELECT
  'fact_transferencias.id_tramite (PK)' AS columna,
  COUNT(*) - COUNT(id_tramite) AS nulos,
  CASE WHEN COUNT(*) = COUNT(id_tramite) THEN '✓ OK' ELSE '✗ HAY NULOS' END AS resultado
FROM workspace.tp_dnrpa_gold.fact_transferencias

UNION ALL

SELECT
  'fact_transferencias.automotor_marca_codigo (FK)' AS columna,
  COUNT(*) - COUNT(automotor_marca_codigo) AS nulos,
  CASE WHEN COUNT(*) = COUNT(automotor_marca_codigo) THEN '✓ OK' ELSE '⚠ HAY NULOS' END AS resultado
FROM workspace.tp_dnrpa_gold.fact_transferencias

UNION ALL

SELECT
  'fact_transferencias.automotor_tipo_codigo (FK)' AS columna,
  COUNT(*) - COUNT(automotor_tipo_codigo) AS nulos,
  CASE WHEN COUNT(*) = COUNT(automotor_tipo_codigo) THEN '✓ OK' ELSE '⚠ HAY NULOS' END AS resultado
FROM workspace.tp_dnrpa_gold.fact_transferencias

UNION ALL

SELECT
  'fact_transferencias.automotor_modelo_codigo (FK)' AS columna,
  COUNT(*) - COUNT(automotor_modelo_codigo) AS nulos,
  CASE WHEN COUNT(*) = COUNT(automotor_modelo_codigo) THEN '✓ OK' ELSE '⚠ HAY NULOS' END AS resultado
FROM workspace.tp_dnrpa_gold.fact_transferencias

UNION ALL

SELECT
  'fact_transferencias.registro_seccional_codigo (FK)' AS columna,
  COUNT(*) - COUNT(registro_seccional_codigo) AS nulos,
  CASE WHEN COUNT(*) = COUNT(registro_seccional_codigo) THEN '✓ OK' ELSE '⚠ HAY NULOS' END AS resultado
FROM workspace.tp_dnrpa_gold.fact_transferencias;

In [0]:
%sql
-- Validación 4: Verificar integridad referencial entre fact y dimensiones
-- Detecta registros en fact_transferencias que apuntan a códigos inexistentes en las dimensiones

WITH orphan_checks AS (
  SELECT
    'automotor_marca_codigo' AS foreign_key,
    COUNT(DISTINCT f.automotor_marca_codigo) AS codigos_en_fact,
    (
      SELECT COUNT(DISTINCT f2.automotor_marca_codigo)
      FROM workspace.tp_dnrpa_gold.fact_transferencias f2
      LEFT JOIN workspace.tp_dnrpa_gold.dim_marca d ON f2.automotor_marca_codigo = d.automotor_marca_codigo
      WHERE d.automotor_marca_codigo IS NULL AND f2.automotor_marca_codigo IS NOT NULL
    ) AS orphan_records
  FROM workspace.tp_dnrpa_gold.fact_transferencias f
  
  UNION ALL
  
  SELECT
    'automotor_tipo_codigo' AS foreign_key,
    COUNT(DISTINCT f.automotor_tipo_codigo) AS codigos_en_fact,
    (
      SELECT COUNT(DISTINCT f2.automotor_tipo_codigo)
      FROM workspace.tp_dnrpa_gold.fact_transferencias f2
      LEFT JOIN workspace.tp_dnrpa_gold.dim_tipo_vehiculo d ON f2.automotor_tipo_codigo = d.automotor_tipo_codigo
      WHERE d.automotor_tipo_codigo IS NULL AND f2.automotor_tipo_codigo IS NOT NULL
    ) AS orphan_records
  FROM workspace.tp_dnrpa_gold.fact_transferencias f
  
  UNION ALL
  
  SELECT
    'automotor_modelo_codigo' AS foreign_key,
    COUNT(DISTINCT f.automotor_modelo_codigo) AS codigos_en_fact,
    (
      SELECT COUNT(DISTINCT f2.automotor_modelo_codigo)
      FROM workspace.tp_dnrpa_gold.fact_transferencias f2
      LEFT JOIN workspace.tp_dnrpa_gold.dim_modelo d ON f2.automotor_modelo_codigo = d.automotor_modelo_codigo
      WHERE d.automotor_modelo_codigo IS NULL AND f2.automotor_modelo_codigo IS NOT NULL
    ) AS orphan_records
  FROM workspace.tp_dnrpa_gold.fact_transferencias f
  
  UNION ALL
  
  SELECT
    'registro_seccional_codigo' AS foreign_key,
    COUNT(DISTINCT f.registro_seccional_codigo) AS codigos_en_fact,
    (
      SELECT COUNT(DISTINCT f2.registro_seccional_codigo)
      FROM workspace.tp_dnrpa_gold.fact_transferencias f2
      LEFT JOIN workspace.tp_dnrpa_gold.dim_geografia d ON f2.registro_seccional_codigo = d.registro_seccional_codigo
      WHERE d.registro_seccional_codigo IS NULL AND f2.registro_seccional_codigo IS NOT NULL
    ) AS orphan_records
  FROM workspace.tp_dnrpa_gold.fact_transferencias f
)
SELECT
  foreign_key,
  codigos_en_fact,
  orphan_records,
  CASE WHEN orphan_records = 0 THEN '✓ OK' ELSE '✗ HAY REGISTROS HUÉRFANOS' END AS resultado
FROM orphan_checks;

In [0]:
%sql
-- Validación 5: Reconciliación de conteos entre Silver y Gold
-- Verifica que no se hayan perdido registros en la transformación

WITH conteos AS (
  SELECT
    (SELECT COUNT(*) FROM workspace.tp_dnrpa_silver.silver_transferencias WHERE id_tramite IS NOT NULL) AS registros_silver,
    (SELECT COUNT(*) FROM workspace.tp_dnrpa_gold.fact_transferencias) AS registros_gold
)
SELECT
  registros_silver,
  registros_gold,
  registros_silver - registros_gold AS diferencia,
  ROUND(TRY_DIVIDE(registros_gold * 100.0, registros_silver), 2) AS porcentaje_carga,
  CASE 
    WHEN registros_silver = 0 THEN '✗ SIN DATOS EN SILVER'
    WHEN registros_silver = registros_gold THEN '✓ COINCIDENCIA EXACTA'
    WHEN registros_gold >= (registros_silver * 0.99) THEN '⚠ COINCIDENCIA ACEPTABLE (>99%)'
    ELSE '✗ DISCREPANCIA SIGNIFICATIVA'
  END AS resultado
FROM conteos;

## 🔍 Investigación de Duplicados en dim_geografia

La validación detectó que `dim_geografia` tiene 846 filas pero solo 842 códigos únicos.
Esto significa que hay 4 códigos de registro seccional que aparecen más de una vez.

Posibles causas:
* Mismo código asociado a diferentes provincias (error de datos)
* Variaciones en el nombre de la provincia (ej: "BUENOS AIRES" vs "Bs. As.")
* Datos inconsistentes en la fuente

In [0]:
%sql
-- Identificar qué códigos de registro seccional están duplicados
-- y cuántas veces aparecen con qué provincias

SELECT 
  registro_seccional_codigo,
  COUNT(*) AS veces_aparece,
  COUNT(DISTINCT registro_seccional_provincia) AS provincias_distintas,
  COLLECT_SET(registro_seccional_provincia) AS provincias
FROM workspace.tp_dnrpa_gold.dim_geografia
GROUP BY registro_seccional_codigo
HAVING COUNT(*) > 1
ORDER BY veces_aparece DESC, registro_seccional_codigo;

In [0]:
%sql
-- Ver todos los registros duplicados con sus detalles

WITH codigos_duplicados AS (
  SELECT registro_seccional_codigo
  FROM workspace.tp_dnrpa_gold.dim_geografia
  GROUP BY registro_seccional_codigo
  HAVING COUNT(*) > 1
)
SELECT 
  g.registro_seccional_codigo,
  g.registro_seccional_provincia,
  LENGTH(g.registro_seccional_provincia) AS long_provincia,
  HEX(g.registro_seccional_provincia) AS hex_provincia
FROM workspace.tp_dnrpa_gold.dim_geografia g
INNER JOIN codigos_duplicados cd ON g.registro_seccional_codigo = cd.registro_seccional_codigo
ORDER BY g.registro_seccional_codigo, g.registro_seccional_provincia;

In [0]:
%sql
-- Verificar cuántos registros distintos hay en Silver para estos códigos duplicados

WITH codigos_duplicados AS (
  SELECT registro_seccional_codigo
  FROM workspace.tp_dnrpa_gold.dim_geografia
  GROUP BY registro_seccional_codigo
  HAVING COUNT(*) > 1
)
SELECT 
  s.registro_seccional_codigo,
  COUNT(DISTINCT s.registro_seccional_provincia) AS provincias_distintas_silver,
  COLLECT_SET(DISTINCT s.registro_seccional_provincia) AS provincias_silver,
  COUNT(*) AS registros_en_silver
FROM workspace.tp_dnrpa_silver.silver_transferencias s
INNER JOIN codigos_duplicados cd ON s.registro_seccional_codigo = cd.registro_seccional_codigo
GROUP BY s.registro_seccional_codigo
ORDER BY s.registro_seccional_codigo;

## 📊 Vista Analítica - Consultas de Negocio

La vista `vw_transferencias_analitica` desnormaliza automáticamente el modelo estrella:
* Fact_transferencias + 4 dimensiones
* Sin necesidad de escribir JOINs manualmente
* Lista para herramientas de BI (Power BI, Tableau, Looker)

**Casos de uso**:
* Análisis temporal de transferencias
* Rankings por marca/modelo/provincia
* Distribución geográfica
* Tendencias y patrones de mercado

In [0]:
%sql
-- Top 10 marcas más transferidas por provincia
-- Útil para análisis regional de preferencias de marca

WITH totales AS (
  SELECT 
    registro_seccional_provincia,
    automotor_marca_descripcion,
    COUNT(*) as total_transferencias
  FROM workspace.tp_dnrpa_gold.vw_transferencias_analitica
  WHERE registro_seccional_provincia IS NOT NULL
    AND automotor_marca_descripcion IS NOT NULL
  GROUP BY registro_seccional_provincia, automotor_marca_descripcion
),
ranking AS (
  SELECT
    *,
    ROW_NUMBER() OVER (PARTITION BY registro_seccional_provincia ORDER BY total_transferencias DESC) as rn,
    SUM(total_transferencias) OVER (PARTITION BY registro_seccional_provincia) as total_provincia
  FROM totales
)
SELECT 
  registro_seccional_provincia as provincia,
  automotor_marca_descripcion as marca,
  total_transferencias,
  ROUND(total_transferencias * 100.0 / total_provincia, 2) as porcentaje_provincia
FROM ranking
WHERE rn <= 10
ORDER BY provincia, total_transferencias DESC
LIMIT 50;

In [0]:
%sql
-- Evolución mensual de transferencias por tipo de vehículo
-- Detecta tendencias y estacionalidad

SELECT 
  anio_tramite,
  mes_tramite,
  automotor_tipo_descripcion,
  COUNT(*) as transferencias,
  COUNT(DISTINCT automotor_marca_codigo) as marcas_distintas
FROM workspace.tp_dnrpa_gold.vw_transferencias_analitica
WHERE anio_tramite >= 2020
  AND automotor_tipo_descripcion IS NOT NULL
GROUP BY anio_tramite, mes_tramite, automotor_tipo_descripcion
ORDER BY anio_tramite DESC, mes_tramite DESC, transferencias DESC
LIMIT 100;

In [0]:
%sql
-- Top 20 modelos más transferidos con desglose de marca y tipo
-- Identifica los vehículos más populares del mercado

SELECT 
  ROW_NUMBER() OVER (ORDER BY COUNT(*) DESC) as ranking,
  automotor_marca_descripcion as marca,
  automotor_modelo_descripcion as modelo,
  automotor_tipo_descripcion as tipo,
  COUNT(*) as total_transferencias,
  COUNT(DISTINCT registro_seccional_provincia) as provincias_con_ventas,
  ROUND(AVG(CAST(automotor_anio_modelo AS INT)), 0) as anio_promedio
FROM workspace.tp_dnrpa_gold.vw_transferencias_analitica
WHERE automotor_modelo_descripcion IS NOT NULL
  AND automotor_marca_descripcion IS NOT NULL
GROUP BY automotor_marca_descripcion, automotor_modelo_descripcion, automotor_tipo_descripcion
ORDER BY total_transferencias DESC
LIMIT 20;

# 🎉 Resumen Final - Arquitectura Medallion DNRPA

## ✅ Pipeline Completo y Validado

### 📊 Volumetría

```
Bronze:  3,300,000 registros crudos (CSV + Excel)
    ↓
Silver:  2,418,741 registros limpios y normalizados
    ↓
Gold:    2,418,741 hechos + 12,540 dimensiones
    ↓
Vista:   Modelo desnormalizado listo para BI
```

---

### 📚 Notebooks DDL (Setup de Infraestructura)

| Notebook | Path | Contenido |
|----------|------|----------|
| **[DDL_Bronze](#notebook-3465291481176724)** | `/01_DDL/DDL_Bronze` | Esquema + 2 tablas |
| **[DDL_Silver](#notebook-3465291481176725)** | `/01_DDL/DDL_Silver` | Esquema + 1 tabla |
| **[DDL_Gold](#notebook-3465291481176726)** | `/01_DDL/DDL_Gold` | Esquema + 5 tablas + **vista analítica** ✓ |

---

### 🔄 Notebooks ETL (Carga de Datos)

| Notebook | Path | Función |
|----------|------|----------|
| **[02_Bronze_dnrpa](#notebook-860821664547612)** | `/02_Bronze/02_Bronze_dnrpa` | INSERT OVERWRITE desde CSV/Excel |
| **[02_Silver_dnrpa](#notebook-860821664547610)** | `/03_Silver/02_Silver_dnrpa` | Transformaciones PySpark + .saveAsTable() |
| **[04_Gold](#notebook-860821664547601)** | `/04_Gold/04_Gold` (este notebook) | INSERT OVERWRITE 5 tablas + validaciones |

---

### ✅ Validaciones de Calidad (Todas Pasadas)

| Validación | Resultado |
|------------|----------|
| 1. Conteo de registros | ✓ OK - 2.4M hechos, 12.5K dimensiones |
| 2. Unicidad de PKs | ✓ OK - Sin duplicados |
| 3. Valores nulos en PKs/FKs | ✓ OK - 0 nulos |
| 4. Integridad referencial | ✓ OK - 0 huérfanos |
| 5. Reconciliación Silver-Gold | ✓ OK - 100% coincidencia |

---

### 📊 Vista Analítica: `vw_transferencias_analitica`

**Estructura**: Fact + 4 dimensiones con LEFT JOIN automático

**Columnas disponibles**:
* **Temporal**: tramite_fecha, anio_tramite, mes_tramite
* **Geográfica**: registro_seccional_codigo, registro_seccional_provincia
* **Marca**: automotor_marca_codigo, automotor_marca_descripcion
* **Tipo**: automotor_tipo_codigo, automotor_tipo_descripcion
* **Modelo**: automotor_modelo_codigo, automotor_modelo_descripcion
* **Métricas**: automotor_anio_modelo, id_tramite

**Casos de uso demostrados**:
1. ✅ Top 10 marcas por provincia (con % market share)
2. ✅ Evolución mensual por tipo de vehículo
3. ✅ Ranking de modelos más transferidos

---

### 🔧 Mejoras Aplicadas

| Problema Original | Solución Aplicada |
|-------------------|--------------------|
| Silver vacía (0 registros) | Filtro corregido: `F.col("tramite") == "TRANSFERENCIA"` |
| Error CAST en fechas | Cambiado a `TRY_CAST` para manejar valores malformados |
| Duplicados en dim_geografia | Filtro agregado: `AND registro_seccional_provincia IS NOT NULL` |
| DDL mezclado con ETL | Separación completa: notebooks DDL vs notebooks ETL |
| CREATE OR REPLACE lento | Cambiado a `INSERT OVERWRITE` en todos los ETL |

---

### 🚀 Insights de Negocio Obtenidos

**Top 3 Marcas (Nacional)**:
1. 🥇 Volkswagen - 168,904 transferencias (18.56% en Buenos Aires)
2. 🥈 Ford - 128,941 transferencias
3. 🥉 Renault - 110,748 transferencias

**Modelos más populares**:
1. Volkswagen GOL 1.6 (24,060 transferencias)
2. Volkswagen GOL TREND 1.6 (23,725 transferencias)
3. Volkswagen GOL 1.6 (variante 5P, 11,573 transferencias)

**Cobertura geográfica**: 841 registros seccionales en 24 provincias

---

## 🔗 Siguiente Paso: Conectar a BI

La vista `vw_transferencias_analitica` está lista para:
* **Power BI**: Importar como Direct Query
* **Tableau**: Conectar a Databricks SQL Warehouse
* **Looker**: Usar conector Databricks
* **Python/R**: pandas.read_sql() / dbplyr

**Ejemplo de conexión**:
```sql
SELECT * FROM workspace.tp_dnrpa_gold.vw_transferencias_analitica
WHERE anio_tramite >= 2020;
```

---

## 📝 Documentación Completa

Ver **[DDL_Bronze](#notebook-3465291481176724)** para la guía completa de verificación paso a paso.